# Nonisothermal CSTR: steady states and coolant temperature

Exploratory notebook. Written quickly to answer one question, kept because it
worked. This is the starting point for Workshop 1, not a finished product.

## The reactor

A jacketed continuous stirred-tank reactor running the irreversible, exothermic,
first-order liquid-phase reaction

$$ \mathrm{A} \rightarrow \mathrm{B}, \qquad r = k(T)\, C_A,
   \qquad k(T) = k_0 \exp\!\left(-\frac{E}{RT}\right) $$

Feed enters at $C_{Af}$ and $T_f$; a jacket exchanges heat with coolant held at
$T_c$.

## Steady-state balances

Material balance on A over the liquid volume $V$ (mol/min):

$$ 0 = q\,(C_{Af} - C_A) - V\,k(T)\,C_A $$

Energy balance, contents at the outlet temperature $T$ (J/min):

$$ 0 = q\,\rho\,C_p\,(T_f - T) \;+\; (-\Delta H_{\mathrm{rxn}})\,V\,k(T)\,C_A
       \;-\; UA\,(T - T_c) $$

**Sign conventions.** $\Delta H_{\mathrm{rxn}}$ is negative for an exothermic
reaction, so $(-\Delta H_{\mathrm{rxn}})$ is a positive heat release. The
exchange term is written as heat *removed* to the coolant and is therefore
subtracted; it becomes a heat input when $T < T_c$.

## Assumptions

1. Perfect mixing — outlet composition and temperature equal bulk values.
2. Constant density $\rho$ and heat capacity $C_p$.
3. Constant liquid volume $V$.
4. One irreversible reaction, first order in A, no side or reverse reactions.
5. **Constant coolant temperature $T_c$** — the jacket is not modeled as a
   dynamic or distributed system, so jacket hold-up and coolant temperature rise
   are neglected.
6. Negligible heat of mixing, shaft work, and heat loss other than through $UA$.
7. $UA$, $\rho$, $C_p$, and $\Delta H_{\mathrm{rxn}}$ independent of temperature.

## Units

| Symbol | Meaning | Unit |
|---|---|---|
| $q$ | volumetric feed flow rate | L/min |
| $V$ | reactor liquid volume | L |
| $C_A$, $C_{Af}$ | concentration of A in reactor, feed | mol/L |
| $T$, $T_f$, $T_c$ | reactor, feed, coolant temperature | K |
| $\rho$ | liquid density | g/L |
| $C_p$ | liquid heat capacity | J/(g K) |
| $\Delta H_{\mathrm{rxn}}$ | enthalpy of reaction | J/mol |
| $E/R$ | activation energy over gas constant | K |
| $k_0$ | Arrhenius pre-exponential factor | 1/min |
| $UA$ | heat-transfer coefficient times area | J/(min K) |

Every temperature in this notebook is absolute. There are no Celsius values.

> **Running this notebook.** Start Jupyter from the project root
> (`jupyter lab`), not from inside `notebooks/`. The output paths near the end
> are written relative to the working directory.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import fsolve

print("numpy", np.__version__, "| pandas", pd.__version__)

## Nominal parameters

Same numbers as `data/reactor_parameters.yml`. Typed in here as well so the
notebook runs on its own.

In [ ]:
q = 100.0        # L/min
V = 100.0        # L
CAf = 1.0        # mol/L
Tf = 320.0       # K
rho = 1000.0     # g/L
Cp = 0.239       # J/(g K)
dHr = -5.0e4     # J/mol, negative because the reaction is exothermic
EoverR = 8750.0  # K
k0 = 7.2e10      # 1/min
UA = 3.5e4       # J/(min K)
Tc = 300.0       # K

params = dict(q=q, V=V, CAf=CAf, Tf=Tf, rho=rho, Cp=Cp, dHr=dHr,
              EoverR=EoverR, k0=k0, UA=UA, Tc=Tc)
params

## Rate and residuals

`residuals` returns both balances at a candidate state `(CA, T)`. Both are zero
at a steady state.

In [ ]:
def rate_constant(T):
    return k0 * np.exp(-EoverR / T)


def residuals(x, Tc_local=Tc):
    CA, T = x
    r = rate_constant(T) * CA
    material = q * (CAf - CA) - V * r
    energy = (q * rho * Cp * (Tf - T)
              + (-dHr) * V * r
              - UA * (T - Tc_local))
    return [material, energy]


# quick sanity check: the feed state should not already be a steady state
print(residuals([CAf, Tf]))

## One nominal solution

`fsolve` needs a starting point. Try an obvious one.

In [ ]:
guess = [1.0, 300.0]
sol, info, ier, msg = fsolve(residuals, guess, full_output=True)
CA_sol, T_sol = sol
print("flag", ier, msg.strip()[:60])
print(f"CA = {CA_sol:.6f} mol/L")
print(f"X  = {(CAf - CA_sol) / CAf:.6f}")
print(f"T  = {T_sol:.4f} K")
print(f"residual norm = {np.linalg.norm(residuals(sol)):.3e}")

That converged, and the residual norm is tiny. It is tempting to stop here.

But an exothermic reaction in a cooled CSTR can have more than one steady state
at the same operating condition, and `fsolve` returns whichever one its starting
point leads to. One converged solve is evidence that *a* steady state exists at
these conditions — not that it is the only one, and not that a real reactor
would sit there.

So: try many starting points and see how many *different* answers come back.

In [ ]:
# Start from a spread of temperatures. For each one, take the concentration that
# satisfies the material balance exactly at that temperature, so the guess is
# already consistent with one of the two equations.
guess_temperatures = np.arange(300.0, 441.0, 5.0)

TOL_T = 1e-2        # K; two roots closer than this are the same steady state
TOL_RESIDUAL = 1e-6  # accept a solve only below this residual norm


def distinct_steady_states(Tc_local=Tc):
    found = []
    for T0 in guess_temperatures:
        CA0 = q * CAf / (q + V * k0 * np.exp(-EoverR / T0))
        s, info, ier, msg = fsolve(residuals, [CA0, T0], args=(Tc_local,),
                                   full_output=True)
        rnorm = np.linalg.norm(residuals(s, Tc_local))
        if ier != 1 or rnorm > TOL_RESIDUAL:
            continue
        if not (-1e-8 <= s[0] <= CAf + 1e-8) or s[1] <= 0:
            continue
        if any(abs(s[1] - kept["T"]) < TOL_T for kept in found):
            continue
        found.append({"CA": float(s[0]),
                      "conversion": float((CAf - s[0]) / CAf),
                      "T": float(s[1]),
                      "residual_norm": float(rnorm)})
    return sorted(found, key=lambda d: d["T"])


nominal_states = distinct_steady_states()
print(f"{len(nominal_states)} distinct steady states at Tc = {Tc:g} K\n")
for i, s in enumerate(nominal_states):
    print(f"  branch {i}:  CA = {s['CA']:.6f} mol/L   X = {s['conversion']:.6f}"
          f"   T = {s['T']:.4f} K   |F| = {s['residual_norm']:.2e}")

Three, not one. The first solve found the coldest of them.

## Sweep the coolant temperature

How does that change as the jacket gets warmer or colder?

In [ ]:
Tc_values = np.arange(285.0, 315.0 + 1e-9, 0.5)

rows = []
for Tc_i in Tc_values:
    states = distinct_steady_states(Tc_i)
    for branch_index, s in enumerate(states):
        rows.append({"Tc": float(Tc_i),
                     "branch_index": branch_index,
                     "n_steady_states": len(states),
                     "CA": s["CA"],
                     "conversion": s["conversion"],
                     "T": s["T"],
                     "residual_norm": s["residual_norm"]})

sweep_df = pd.DataFrame(rows)
print(len(sweep_df), "rows for", len(Tc_values), "coolant temperatures")

counts = sweep_df.groupby("Tc")["n_steady_states"].first()
multi = counts[counts >= 3]
print(f"three steady states at {len(multi)} of {len(counts)} conditions, "
      f"for Tc between {multi.index.min():g} and {multi.index.max():g} K")
sweep_df.head()

## Figure

Plot each branch separately. Joining them into one line across a turning point
would draw a curve the model does not have.

In [ ]:
fig, ax = plt.subplots(figsize=(5.0, 3.6))
labels = {0: "lower branch", 1: "middle branch", 2: "upper branch"}
for idx, group in sweep_df.groupby("branch_index"):
    g = group.sort_values("Tc")
    ax.plot(g["Tc"], g["T"], marker="o", markersize=3, linewidth=1.2,
            label=labels.get(int(idx), f"branch {idx}"))
ax.set_xlabel("Coolant temperature Tc (K)")
ax.set_ylabel("Reactor temperature T (K)")
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()

# Paths are relative to the working directory -- start Jupyter from the project
# root or this will write somewhere surprising (or fail).
sweep_df.to_csv("results/steady_states.csv", index=False, float_format="%.10g")
fig.savefig("results/steady_state_locus.png", dpi=150)
print("wrote results/steady_states.csv and results/steady_state_locus.png")

## What this shows, and what it does not

**What the computation supports.** Over a range of coolant temperatures — here
roughly 290.5 K to 311 K at the 0.5 K resolution of this sweep — the steady-state
balances have three solutions rather than one. Outside that range there is one.
The locus is S-shaped: the upper and lower branches are connected through a
middle branch, and the branch found by a solver depends entirely on where it
starts.

**What it does not support.**

- *Nothing here is a stability result.* These are steady states of the algebraic
  balances. Whether a branch is stable is a question about the **dynamic**
  model — the transient material and energy balances and the eigenvalues of
  their Jacobian — and this notebook never writes those equations down. The
  middle branch of an S-shaped curve is often unstable in systems like this, but
  "often, in systems like this" is not evidence about *this* system. Saying the
  middle branch is unstable requires a calculation that has not been done.
- *Nothing here is an ignition or extinction prediction.* Those are statements
  about what happens when an operating parameter is moved slowly past a turning
  point, which again needs the dynamic model.
- *The edges of the multiplicity range are resolved only to 0.5 K*, the sweep
  step. The turning points are somewhere inside those intervals.
- *The parameters are a teaching set*, adjusted to make the behavior visible.
  They are not measurements of a particular reactor, so quantitative agreement
  with a published case would be a coincidence unless the parameters match.

Workshop 2 revisits every one of these statements against the literature. Expect
at least one of them to need narrowing.